In [0]:
from datetime import datetime
now = datetime.now()
today_date = now.strftime("%Y-%m-%d")
dbutils.widgets.text('start_date','2000-01-01',"Starting date")
dbutils.widgets.text('end_date',today_date,"Ending date")
dbutils.widgets.text('LAT','30.4745',"Latitude")
dbutils.widgets.text('LON','78.1551',"Longitude")

In [0]:
import requests
import pandas as pd






def fetch_historical_weather(LAT=30.4745, LON=78.1551, start_date="2000-01-01", end_date=today_date):
    # Example Mussoorie Coordinates
    # LAT = 30.4745
    # LON = 78.1551
    
    # The exact parameters from your PySpark schema
    METRICS = "temperature_2m,relative_humidity_2m,cloud_cover_mid,cloud_cover_high,cloud_cover_low,rain,snowfall,precipitation,dew_point_2m,wind_speed_100m,wind_speed_10m,wind_direction_10m"
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "start_date": start_date,
	    "end_date": end_date,
        "latitude": LAT,
        "longitude": LON,
        "hourly": METRICS,
        "timezone": "auto",
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # Load directly into a Pandas DataFrame using the JSON dictionary
    df_live = pd.DataFrame(data["hourly"])
    
    return df_live

df_old_data = fetch_historical_weather()
df_old_data['time'] = pd.to_datetime(df_old_data['time'])

In [0]:
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType, IntegerType
from pyspark.sql import functions as F

# 1. Define your exact schema
schema = StructType([
    StructField("time", TimestampType(), True),
    StructField("temperature_2m", DoubleType(), True),
    StructField("relative_humidity_2m", IntegerType(), True),
    StructField("cloud_cover_mid", IntegerType(), True),
    StructField("cloud_cover_high", IntegerType(), True),
    StructField("cloud_cover_low", IntegerType(), True),
    StructField("rain", DoubleType(), True),
    StructField("snowfall", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("dew_point_2m", DoubleType(), True),
    StructField("wind_speed_100m", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("wind_direction_10m", IntegerType(), True),
])

df_old_data = spark.createDataFrame(df_old_data, schema=schema)
df_old_data = df_old_data.withColumn('read_timestamp', F.current_timestamp())




In [0]:
display(df_old_data.limit(10))

In [0]:
# display(df_old_data.dtypes)

Bronze Layer 
Loading the CSV/API response as-is with read_timestamp and _metadata columns.

In [0]:
# path = f's3://weather-80-25/mussoorie_1980_2026.csv' 

In [0]:
# from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType, IntegerType
# from pyspark.sql import functions as F

# schema = StructType([
#     StructField("time", TimestampType(), True),
#     StructField("temperature_2m", DoubleType(), True),
#     StructField("relative_humidity_2m", IntegerType(), True),
#     StructField("cloud_cover_mid", IntegerType(), True),
#     StructField("cloud_cover_high", IntegerType(), True),
#     StructField("cloud_cover_low", IntegerType(), True),
#     StructField("rain", DoubleType(), True),
#     StructField("snowfall", DoubleType(), True),
#     StructField("precipitation", DoubleType(), True),
#     StructField("dew_point_2m", DoubleType(), True),
#     StructField("wind_speed_100m", DoubleType(), True),
#     StructField("wind_speed_10m", DoubleType(), True),
#     StructField("wind_direction_10m", IntegerType(), True),
# ])



# def weather_fetch(spark, path, schema):
    
#     df = (
#         spark.createDataFrame(df_old_data, schema)
#         .option('header', True)
#         .schema(schema)
#         .load(path)
#         .withColumn('read_timestamp', F.current_timestamp())
#         .select('*', '_metadata.file_name', '_metadata.file_size')
#     )
    
#     return df

# my_weather_df = weather_fetch(spark, path, schema)
# display(my_weather_df.limit(10))

Saving in memory data frame as permanent delta table 

In [0]:
%sql 
-- 1. Creating the main project catalog
CREATE CATALOG IF NOT EXISTS weather;

-- 2. Creating 3 schema inside the weather catalog
CREATE SCHEMA IF NOT EXISTS weather.bronze;
CREATE SCHEMA IF NOT EXISTS weather.silver;
CREATE SCHEMA IF NOT EXISTS weather.gold;



In [0]:
df_old_data.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather.bronze.bronze_weather")

In [0]:

# def clean_silver_weather_data(df: DataFrame) -> DataFrame:
    
#     df = df.withColumn("date", to_date(col("time")))
    
#     df = df.withColumn(
#         "season",
#         when(month(col("time")).isin(12, 1, 2), "WR")
#         .when(month(col("time")).isin(3, 4, 5), "SG")
#         .when(month(col("time")).isin(6, 7, 8), "SR")
#         .when(month(col("time")).isin(9, 10, 11), "AU")
#     )
    
#     return df

In [0]:
# invalid_condition = ~(
#     (F.col("wind_direction_10m").between(0, 360)) &
#     (F.col("relative_humidity_2m").between(0, 101)) &
#     (F.col("cloud_cover_mid").between(0, 100)) &
#     (F.col("cloud_cover_high").between(0, 100)) &
#     (F.col("cloud_cover_low").between(0, 100)) &
#     (F.col("temperature_2m").between(-30, 50)) &
#     (F.col("wind_speed_10m") >= 0) &
#     (F.col("wind_speed_100m") >= 0) &
#     (F.col("rain") >= 0) &
#     (F.col("precipitation") >= 0) &
#     (F.col("snowfall") >= 0) 
#     # (F.col("dew_point_2m") <= F.col("temperature_2m"))
# )

# display(df_silver.filter(invalid_condition))

counting missing value in every column  

In [0]:
# from pyspark.sql.functions import col, sum

# # Count null values for each column and display the result
# missing_counts = df_silver.select([
#     sum(col(c).isNull().cast("int")).alias(c) for c in df_silver.columns
# ])

# display(missing_counts)

In [0]:
# # adding a decade partition
# from pyspark.sql.functions import floor
# df_silver = df_silver.withColumn(
#     "decade_partition", 
#     (floor(col("year") / 10) * 10).cast("int")
# )

**The Golden Layer**
The Gold layer is completely focused on the end-user or the specific business problem. Data here is usually heavily aggregated, summarized, and joined with other tables. You rarely look at individual hourly readings in the Gold layer.

In [0]:


# df_gold_daily = (
#     df_valid.groupBy("year", "month", "day")
#     .agg(
#         F.avg("temperature_2m").alias("avg_temp"),
#         F.min("temperature_2m").alias("min_temp"),
#         F.max("temperature_2m").alias("max_temp"),
#         F.sum("rain").alias("total_rain"),
#         F.sum("snowfall").alias("total_snowfall"),
#         F.avg("relative_humidity_2m").alias("avg_humidity"),
#         F.avg("wind_speed_10m").alias("avg_wind_speed"),
#         F.avg("cloud_cover_mid").alias("avg_cloud_cover_mid")
#     )
# )

In [0]:
# In Databricks, 'catalog.schema.weather_data_processing' refers to the table 'weather_data_processing'
# in the specified 'schema' within a particular 'catalog'. 'schema.weather_data_processing' refers to the
# table in the specified 'schema' within the current catalog. The catalog is a higher-level namespace,
# so omitting it uses the default catalog set for your session.